In [0]:
%sql
-- Total revenue, profit, and orders (all time)
SELECT
    SUM(gross_revenue) AS total_revenue,
    SUM(total_profit)  AS total_profit,
    ROUND(SUM(total_profit) / SUM(gross_revenue) * 100, 2) AS overall_margin_pct
FROM workspace.gold.daily_sales_summary;

In [0]:
%sql
-- This month vs last month revenue (trend indicator)
SELECT
    order_year,
    order_month,
    SUM(gross_revenue) AS monthly_revenue
FROM workspace.gold.monthly_category_performance
GROUP BY order_year, order_month
ORDER BY order_year DESC, order_month DESC
LIMIT 2;

In [0]:
%sql
/*Revenue trend chart (line chart over time)*/

SELECT
    order_date,
    SUM(gross_revenue) AS daily_revenue,
    SUM(total_profit)  AS daily_profit
FROM workspace.gold.daily_sales_summary
GROUP BY order_date
ORDER BY order_date;

In [0]:
%sql

/*Category performance (bar chart)*/
SELECT
    category,
    SUM(gross_revenue) AS total_revenue,
    SUM(total_profit)  AS total_profit,
    ROUND(AVG(margin_pct), 2) AS avg_margin_pct
FROM workspace.gold.monthly_category_performance
GROUP BY category
ORDER BY total_revenue DESC;

In [0]:
%sql
--Top 10 customers (table/leaderboard)
SELECT
    customer_id,
    region,
    customer_segment,
    total_orders,
    lifetime_revenue
FROM workspace.gold.top_customers
ORDER BY lifetime_revenue DESC
LIMIT 10;

In [0]:
%sql
--Revenue by region + segment (pivot-style breakdown)

SELECT
    region,
    customer_segment,
    COUNT(*) AS customer_count,
    SUM(lifetime_revenue) AS total_revenue,
    ROUND(AVG(lifetime_revenue), 2) AS avg_revenue_per_customer
FROM workspace.gold.top_customers
GROUP BY region, customer_segment
ORDER BY total_revenue DESC;

In [0]:
%sql
--Month-over-month growth (window function)
SELECT
    order_year,
    order_month,
    category,
    gross_revenue,
    LAG(gross_revenue) OVER (
        PARTITION BY category ORDER BY order_year, order_month
    ) AS prev_month_revenue,
    ROUND(
        (gross_revenue - LAG(gross_revenue) OVER (
            PARTITION BY category ORDER BY order_year, order_month
        )) / LAG(gross_revenue) OVER (
            PARTITION BY category ORDER BY order_year, order_month
        ) * 100, 2
    ) AS mom_growth_pct
FROM workspace.gold.monthly_category_performance
ORDER BY category, order_year, order_month;